# Prova dell'agente di viaggio

1. database e indice vettoriale pronti
2. i tool uno per uno, senza LLM
3. l'agente (provider e modello di config.yaml), in una conversazione
4. cosa e' stato salvato nel database

Il kernel deve partire da `backend/` (o da `backend/notebooks/`: la cella sotto lo sistema).

In [1]:
import json
import logging
import os
from pathlib import Path

# i percorsi di .env, travel.db e chroma_data sono relativi a backend/
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("cwd:", Path.cwd())

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
for noisy in ("httpx", "httpcore", "chromadb", "sentence_transformers", "urllib3"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

from sqlalchemy import func, select

from app.core.config import settings
from app.db.session import Base, SessionLocal, engine
from app.models import models
from app.models.models import Activity, Booking, Hotel

print("chat:", settings.llm_provider, "|", {"gemini": settings.gemini_model, "groq": settings.groq_model, "mistral": settings.mistral_model, "huggingface": settings.hf_model}[settings.llm_provider])
print("embedding:", settings.embedding_provider, "|", settings.gemini_embedding_model if settings.embedding_provider == "gemini" else settings.embedding_model)
print("chiavi impostate: Gemini", bool(settings.gemini_api_key), "| Groq", bool(settings.groq_api_key), "| Mistral", bool(settings.mistral_api_key), "| Hugging Face", bool(settings.hf_token))
print("(provider e modelli stanno in config.yaml: dopo una modifica riavvia il kernel)")

cwd: c:\Users\sonia\Documents\Challenge Reply\Virtual-Travel-Assistant\backend
chat: groq | openai/gpt-oss-120b
embedding: huggingface | paraphrase-multilingual-MiniLM-L12-v2
chiavi impostate: Gemini True | Groq True | Hugging Face True
(provider e modelli stanno in config.yaml: dopo una modifica riavvia il kernel)


## 1. Database e indice vettoriale

In [2]:
from app.services.rag import get_collection, index_activities
from scripts.seed import seed

Base.metadata.create_all(engine)
session = SessionLocal()

if session.scalar(select(func.count()).select_from(Hotel)) == 0:
    print("Database vuoto: seed in corso", seed(session))
    print("Attivita' indicizzate:", index_activities(session))
else:
    print("Database gia' popolato:", session.scalar(select(func.count()).select_from(Hotel)), "hotel,",
          session.scalar(select(func.count()).select_from(Activity)), "attivita'")
# l'indice vettoriale e' per provider di embedding: se e' vuoto (primo avvio o provider cambiato) lo riempie
if get_collection().count() == 0:
    print("Indice vuoto (embedding:", settings.embedding_provider + "): indicizzo", index_activities(session), "attivita'")


Database gia' popolato: 24 hotel, 64 attivita'


c:\Users\sonia\Documents\Challenge Reply\Virtual-Travel-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2989.60it/s]


## 2. I tool, uno per uno

`AgentContext` tiene sessione, utente (qui l'utente demo, id 1) e gli ultimi itinerari generati.
Ogni tool si chiama con `.invoke({...})`: e' esattamente quello che fa l'agente.

In [3]:
from app.services.agent_tools import AgentContext, build_tools

ctx = AgentContext(session, user_id=1)
tools = {t.name: t for t in build_tools(ctx)}

def show(raw):
    try:
        print(json.dumps(json.loads(raw), ensure_ascii=False, indent=2))
    except ValueError:
        print(raw)

print(list(tools))

['list_destinations', 'list_hotels', 'find_airport', 'list_flight_dates', 'search_activities', 'generate_itineraries', 'save_proposal']


In [4]:
show(tools["list_destinations"].invoke({}))

{
  "Giappone": [
    "Osaka",
    "Tokyo"
  ],
  "Grecia": [
    "Atene",
    "Santorini"
  ],
  "Portogallo": [
    "Lisbona",
    "Porto"
  ],
  "Spagna": [
    "Barcellona",
    "Madrid"
  ]
}


In [5]:
show(tools["list_hotels"].invoke({"country": "Spagna"}))

[
  {
    "nome": "Albergue Born Urban",
    "citta": "Barcellona",
    "stelle": 2,
    "descrizione": "Alloggio economico nel quartiere Born, ideale per gruppi e famiglie con camere da quattro letti."
  },
  {
    "nome": "Hotel Eixample Garden",
    "citta": "Barcellona",
    "stelle": 4,
    "descrizione": "Hotel elegante nell'Eixample con giardino interno, a pochi passi dalle opere di Gaudí."
  },
  {
    "nome": "Grand Hotel Mar Blau",
    "citta": "Barcellona",
    "stelle": 5,
    "descrizione": "Torre sul mare con piscina panoramica, spa e camere con vista sulla spiaggia."
  },
  {
    "nome": "Hostal Plaza Mayor",
    "citta": "Madrid",
    "stelle": 2,
    "descrizione": "Pensione semplice e pulita a due passi da Plaza Mayor, con camere familiari fino a quattro posti."
  },
  {
    "nome": "Hotel Retiro Park",
    "citta": "Madrid",
    "stelle": 4,
    "descrizione": "Hotel moderno vicino al parco del Retiro, colazione inclusa e camere luminose per due o tre persone."
  },


In [6]:
show(tools["list_flight_dates"].invoke({"home_airport": "MXP", "country": "Spagna", "month": 11}))

{
  "periodo": "2026-11-01 - 2026-11-30",
  "citta": {
    "Barcellona": [
      {
        "giorno": "2026-11-01",
        "prezzo_minimo_a_persona": 69.2
      },
      {
        "giorno": "2026-11-03",
        "prezzo_minimo_a_persona": 69.07
      },
      {
        "giorno": "2026-11-04",
        "prezzo_minimo_a_persona": 60.23
      },
      {
        "giorno": "2026-11-05",
        "prezzo_minimo_a_persona": 50.74
      },
      {
        "giorno": "2026-11-06",
        "prezzo_minimo_a_persona": 54.47
      },
      {
        "giorno": "2026-11-07",
        "prezzo_minimo_a_persona": 70.08
      },
      {
        "giorno": "2026-11-08",
        "prezzo_minimo_a_persona": 69.86
      },
      {
        "giorno": "2026-11-09",
        "prezzo_minimo_a_persona": 63.81
      },
      {
        "giorno": "2026-11-11",
        "prezzo_minimo_a_persona": 74.02
      },
      {
        "giorno": "2026-11-12",
        "prezzo_minimo_a_persona": 54.24
      },
      {
        "giorno": 

In [7]:
show(tools["search_activities"].invoke({"country": "Spagna", "preferences": ["sport", "relax"]}))

[
  {
    "nome": "Percorso spa e bagni termali",
    "citta": "Barcellona",
    "categoria": "relax",
    "prezzo_a_persona": 70.0,
    "affinita": 0.63
  },
  {
    "nome": "Partita di padel con un istruttore",
    "citta": "Madrid",
    "categoria": "sport",
    "prezzo_a_persona": 40.0,
    "affinita": 0.6
  },
  {
    "nome": "Bici e picnic al parco del Retiro",
    "citta": "Madrid",
    "categoria": "relax",
    "prezzo_a_persona": 25.0,
    "affinita": 0.6
  },
  {
    "nome": "Bagni arabi e massaggi",
    "citta": "Madrid",
    "categoria": "relax",
    "prezzo_a_persona": 45.0,
    "affinita": 0.53
  },
  {
    "nome": "Corso di surf e paddle alla Barceloneta",
    "citta": "Barcellona",
    "categoria": "sport",
    "prezzo_a_persona": 60.0,
    "affinita": 0.52
  }
]


### `generate_itineraries`: caso felice e caso di budget troppo basso

In [8]:
args = dict(budget=2500, country="Spagna", activities=["cultura", "relax"], month=11, home_airport="MXP", travelers=2)
show(tools["generate_itineraries"].invoke(args))

INFO app.services.planner: Planner: Spagna, 7 notti, 2 viaggiatori, budget 2500.0: 107 candidati validi, 12 itinerari proposti


[
  {
    "numero": 1,
    "etichetta": "Più adatto a te",
    "citta": "Madrid",
    "date": "2026-11-07 - 2026-11-14 (7 notti)",
    "volo_andata": "07/11 20:00, 68.48 euro a persona",
    "volo_ritorno": "14/11 13:00, 54.29 euro a persona",
    "hotel": "Gran Hotel Castellana (5 stelle), camere: Doppia 1",
    "giorni": [
      {
        "giorno": "2026-11-07",
        "attivita": "Trekking nella Sierra de Guadarrama",
        "categoria": "natura"
      },
      {
        "giorno": "2026-11-08",
        "attivita": "Bici e picnic al parco del Retiro",
        "categoria": "relax"
      },
      {
        "giorno": "2026-11-09",
        "attivita": "Bagni arabi e massaggi",
        "categoria": "relax"
      },
      {
        "giorno": "2026-11-10",
        "attivita": "Tapas tour a La Latina",
        "categoria": "gastronomia"
      },
      {
        "giorno": "2026-11-11",
        "attivita": "Partita di padel con un istruttore",
        "categoria": "sport"
      },
      {
  

In [9]:
show(tools["generate_itineraries"].invoke({**args, "budget": 200}))

INFO app.services.planner: Planner: nessun itinerario per Spagna (over_budget, budget 200.0)


{
  "errore": "over_budget",
  "messaggio": "Con questo budget non riesco a costruire il viaggio: servono almeno 1086.72 euro.",
  "costo_minimo": 1086.72
}


In [10]:
show(tools["generate_itineraries"].invoke({**args, "month": 13}))

Dati non validi: month: Input should be less than or equal to 12


### `save_proposal`

In [11]:
tools["generate_itineraries"].invoke(args)   # ripopola ctx.itineraries
show(tools["save_proposal"].invoke({"itinerary_number": 1}))
show(tools["save_proposal"].invoke({"itinerary_number": 99}))
session.rollback()   # niente da tenere: questa era solo una prova sui tool

INFO app.services.planner: Planner: Spagna, 7 notti, 2 viaggiatori, budget 2500.0: 107 candidati validi, 12 itinerari proposti


{
  "proposta_salvata": 1,
  "stato": "proposed",
  "totale_euro": 2422.02
}
Numero non valido: prima genera gli itinerari e scegli uno dei numeri proposti.


## 3. L'agente

Una conversazione con il provider impostato in `config.yaml` (`llm_provider`). `chat()` mantiene la cronologia, stampa i tool
che l'agente ha chiamato (con gli argomenti) e poi la risposta; un errore del modello non ferma il notebook.
Per provare l'altro provider cambia `llm_provider` in `config.yaml` e riavvia il kernel (oppure `new_agent("gemini")` / `new_agent("groq")` / `new_agent("mistral")` / `new_agent("huggingface")`).


In [12]:
from app.services.agent import build_agent, build_model

def new_agent(provider=None):
    '''Un agente con contesto e cronologia nuovi. `provider` omesso = llm_provider di config.yaml.'''
    global agent, history
    provider = provider or settings.llm_provider
    agent = build_agent(AgentContext(session, user_id=1), build_model(provider))
    history = []
    print("agente pronto:", provider)

def chat(text):
    history.append({"role": "user", "content": text})
    try:
        result = agent.invoke({"messages": history})
    except Exception as error:
        history.pop()
        print()
        print("Utente:", text)
        print()
        print(f"  [ERRORE] {type(error).__name__}: {str(error)[:300]}")
        return
    new = result["messages"][len(history):]   # messaggi nuovi: chiamate ai tool, risultati, risposta
    for message in new:
        for call in getattr(message, "tool_calls", None) or []:
            print(f"  [tool] {call['name']}({call['args']})")
    answer = result["messages"][-1].text
    history.append({"role": "assistant", "content": answer})
    print()
    print("Utente:", text)
    print()
    print("Assistente:", answer)

new_agent()


agente pronto: groq


In [13]:
chat("Ciao! Vorrei fare un viaggio ma non so ancora dove andare.")

  [tool] list_destinations({})

Utente: Ciao! Vorrei fare un viaggio ma non so ancora dove andare.

Assistente: Perfetto, ti propongo alcune mete disponibili:

- **Giappone** (Osaka, Tokyo)  
- **Grecia** (Atene, Santorini)  
- **Portogallo** (Lisbona, Porto)  
- **Spagna** (Barcellona, Madrid)

Per poter cercare le opzioni più adatte, ho bisogno di qualche dettaglio in più:

1. **Qual è il tuo budget totale (in euro) per il viaggio?**  
2. **In quale mese vorresti partire?** (indica il numero: 1 = gennaio, …, 12 = dicembre)

Fammi sapere e poi procederemo con le altre informazioni!


In [14]:
chat("Siamo in 2, ci piacciono relax e cultura. Partiamo da Fiumicino a novembre, abbiamo 3000 euro. Cosa mi consigli?")

  [tool] find_airport({'query': 'Fiumicino'})
  [tool] list_destinations({})

Utente: Siamo in 2, ci piacciono relax e cultura. Partiamo da Fiumicino a novembre, abbiamo 3000 euro. Cosa mi consigli?

Assistente: Ottimo, grazie per le informazioni!  

Al momento ho:

- **Viaggiatori:** 2  
- **Budget totale:** 3 000 €  
- **Mese di partenza:** novembre (11)  
- **Aeroporto di partenza:** FCO (Fiumicino)  
- **Preferenze:** relax e cultura  

Mancano solo due dettagli:

1. **Durata del soggiorno** (numero di notti). Se non hai una preferenza, consideriamo 7 notti.  
2. **Destinazione / città** tra le opzioni disponibili: Giappone (Osaka, Tokyo), Grecia (Atene, Santorini), Portogallo (Lisbona, Porto) o Spagna (Barcellona, Madrid).  

Fammi sapere la durata desiderata e quale città ti ispira di più, così potrò generare gli itinerari più adatti!


In [15]:
chat("Spagna mi piace. Fammi vedere cosa avete.")

INFO groq._base_client: Retrying request to /openai/v1/chat/completions in 13.000000 seconds
INFO groq._base_client: Retrying request to /openai/v1/chat/completions in 2.000000 seconds



Utente: Spagna mi piace. Fammi vedere cosa avete.

Assistente: Perfetto, quindi puntiamo alla **Spagna**.  

Per completare la ricerca ho solo un ultimo dato:

**Quante notti vorreste restare?** (se non hai una preferenza, consideriamo 7 notti).

Fammi sapere e poi ti mostrerò gli itinerari disponibili!


### Modifica richiesta e scelta

In [16]:
chat("Troppo caro, fammi vedere con 5 notti invece di 7.")

INFO groq._base_client: Retrying request to /openai/v1/chat/completions in 14.000000 seconds
INFO app.services.planner: Planner: Spagna, 5 notti, 2 viaggiatori, budget 3000.0: 119 candidati validi, 12 itinerari proposti
INFO groq._base_client: Retrying request to /openai/v1/chat/completions in 44.000000 seconds
INFO groq._base_client: Retrying request to /openai/v1/chat/completions in 1.000000 seconds


  [tool] generate_itineraries({'activities': ['relax', 'cultura'], 'budget': 3000, 'country': 'Spagna', 'home_airport': 'FCO', 'month': 11, 'nights': 5, 'travelers': 2})

Utente: Troppo caro, fammi vedere con 5 notti invece di 7.

Assistente: Ecco gli itinerari che rientrano nel tuo budget di **3 000 €** per 2 persone, 5 notti a novembre in Spagna, con attività di **relax** e **cultura**.

| # | Etichetta | Città | Date (5 notti) | Volo totale (2 pers.) | Hotel (2 pers.) | Attività (cultura + relax) | Costo totale |
|---|-----------|-------|----------------|-----------------------|----------------|----------------------------|--------------|
| **1** | **Più adatto a te** | **Barcellona** | 21 – 26 nov | 245,86 € | Grand Hotel Mar Blau (5★) – camera doppia | Sagrada Familia, Montserrat, Museo Picasso, Spa, serata sulla spiaggia | **1 974,73 €** |
| **2** | **Più economico** | **Madrid** | 04 – 09 nov | 241,28 € | Hostal Plaza Mayor (2★) – camera doppia | Palazzo Reale, Prado & Reina Sof

In [17]:
chat("Prendo il primo. Salvalo.")

INFO groq._base_client: Retrying request to /openai/v1/chat/completions in 21.000000 seconds
INFO groq._base_client: Retrying request to /openai/v1/chat/completions in 26.000000 seconds


  [tool] save_proposal({'itinerary_number': 1})

Utente: Prendo il primo. Salvalo.

Assistente: Perfetto! Ho salvato la **proposta 1** (Barcellona, 5 notti, 1 974,73 €) come **proposta**.  

Ricorda che non è ancora una prenotazione confermata: è solo una bozza che potrai confermare in seguito o modificare se cambi idea.  

Se hai bisogno di ulteriori dettagli (ad esempio informazioni sui voli, sull’hotel o sulle attività) o vuoi procedere con la conferma, fammi sapere!


## 4. Cosa e' stato salvato

In [18]:
session.expire_all()
for b in session.scalars(select(Booking).order_by(Booking.id)):
    print(b.id, b.status, f"{b.total_cost} euro", b.itinerary.get("city"), b.itinerary.get("label"))

1 proposed 1974.73 euro Barcellona Più adatto a te


Per ripulire le proposte di prova: `session.query(Booking).delete(); session.commit()`.